In [ ]:
# Step 1. GCS 연결
from google.colab import auth
auth.authenticate_user()

from google.cloud import storage

PROJECT_ID = "project-id"
BUCKET_NAME = "sign-language-data-2026"

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

print("GCS 연결 성공")

GCS 연결 성공!


In [ ]:
# Step 2. morpheme에서 단어 라벨 추출 (오류 기록 포함) GCS에 저장

import json

label_map = {}
errors = []
count = 0

blobs = bucket.list_blobs(prefix="raw/morpheme/01/")

for blob in blobs:
    count += 1
    try:
        content = blob.download_as_text()
        data = json.loads(content)

        filename = blob.name.split("/")[-1]
        word_id = int(filename.split("_")[2].replace("WORD", ""))
        word_name = data["data"][0]["attributes"][0]["name"]

        # 1. 중복 불일치 확인
        if word_id in label_map and label_map[word_id] != word_name:
            errors.append({
                "type": "중복 불일치",
                "file": blob.name,
                "error": f"WORD{word_id}: {label_map[word_id]} vs {word_name}"
            })

        # 2. 빈 단어명 확인
        if not word_name or word_name.strip() == "":
            errors.append({
                "type": "빈 단어명",
                "file": blob.name,
                "error": f"WORD{word_id} 단어명이 비어있음"
            })

        label_map[word_id] = word_name

    except Exception as e:
        errors.append({
            "type": "파싱 오류",
            "file": blob.name,
            "error": str(e)
        })

    if count % 1000 == 0:
        print(f"[{count}개 처리 중...] 현재 {len(label_map)}개 단어 수집")

# 3. 누락 확인 (WORD1 ~ WORD3000)
expected_ids = set(range(1, 3001))
actual_ids = set(label_map.keys())
missing_ids = expected_ids - actual_ids

print(f"\n총 {len(label_map)}개 단어 추출 완료")
print(f"오류 발생: {len(errors)}개")
print(f"누락된 WORD ID: {len(missing_ids)}개 {sorted(missing_ids) if missing_ids else ''}")
print(f"예시: {dict(list(label_map.items())[:5])}")

if errors:
    print("\n오류 목록:")
    for e in errors:
        print(f"  [{e['type']}] {e['file']} → {e['error']}")


In [ ]:
# 오류 없으면 GCS에 저장
label_map_json = json.dumps(label_map, ensure_ascii=False, indent=2)
blob = bucket.blob("processed/label_map.json")
blob.upload_from_string(label_map_json, content_type="application/json")
print("label_map.json GCS 저장 완료!")